# Experiment 10 — Daily home care nested in professional recall

**Question.** Experiments 07–09 represent plaque between professional visits as a smooth accumulation curve, $u=u_{\max}(1-e^{-k\tau})$, with daily brushing absorbed into its shape. Is that coarse-graining valid? And how does home-care efficacy enter the results?

**Why two plaque pools.** With daily partial removal, a single plaque pool settles into a steady daily cycle within a few days. It cannot produce the slow build-up between recalls that the Exp 07 envelope describes. That build-up comes from plaque that home care misses.

**Plaque model.** The plaque signal is $u=u_a+u_r$:
* **Accessible plaque** $u_a$ regrows within days: $\dot u_a=k_a(U_a-u_a)$ with $1/k_a=2$ units (about 3 days). Once a day, brushing removes a fraction $\eta$ (home-care efficacy): $u_a\to(1-\eta)u_a$.
* **Retained plaque** $u_r$ sits at sites home care misses and accumulates over weeks: $\dot u_r=k_r(U_r-u_r)$ with $k_r=0.05$, the Exp 07 value.
* **Professional recall** (every $T=60$) resets both pools to zero, which is supragingival cleaning. In the debridement variant it also sets $M\to0$, which is subgingival instrumentation.

**Time mapping.** One day is 0.6 units, so one unit is about 1.7 days, $T=60$ is about 100 days (a roughly three-monthly recall) and $1/k_r$ is about 5 weeks. The host relaxes in about 1.7 days, so it can partly follow the daily cycle. That is exactly why the coarse-graining needs testing.

**Comparison.** *Explicit* forcing (daily drops) is compared with *averaged* forcing, in which $u_a$ is replaced by its daily mean interpolated smoothly: the envelope assumed in Exp 07. Results are expressed as the retained-plaque peak before recall, $p_r$. Hosts as before: slow memory ($\varepsilon=0.03, 0.01$) and instant adaptation (same equilibria).

**Sections**
* **A** forcing
* **B** validation: $U_a=0$ reproduces Exp 07; example trajectories
* **C** thresholds and latency, explicit vs averaged, across $\eta$ and $U_a$; collapse on the daily-mean accessible plaque
* **D** debridement at recall with daily brushing
* **E** checks and summary

In [ ]:
# =====================================================================
# EXPERIMENT 10 — folder setup
# =====================================================================
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/Biosemiotic_Perio")
except ImportError:                      # local execution
    BASE_DIR = Path.cwd() / "Biosemiotic_Perio"
EXP_DIR = BASE_DIR / "Experiment_10_Daily_Home_Care"
EXP_DIR.mkdir(parents=True, exist_ok=True)
QUICK = os.environ.get("EXP_QUICK") == "1"     # set True for a fast smoke test with reduced grids
print(f"Experiment folder:\n{EXP_DIR}\nExists: {EXP_DIR.exists()}   QUICK={QUICK}")

In [ ]:
import sys, json, time, platform, shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.optimize import brentq

versions_text = f"""
Experiment: 10 Daily home care nested in professional recall
Created: {datetime.now(timezone.utc).isoformat()}

Python: {sys.version}
Platform: {platform.platform()}
NumPy: {np.__version__}
Pandas: {pd.__version__}
SciPy: {scipy.__version__}
Matplotlib: {matplotlib.__version__}
"""
with open(EXP_DIR / "software_versions.txt", "w") as f:
    f.write(versions_text)
print(versions_text)

# ---- figure style (same tokens as Experiment 07) ------------------------
INK, INK2, GRIDC, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
MEM_COL = {0.3: "#86b6ef", 0.1: "#5598e7", 0.03: "#2a78d6", 0.01: "#1c5cab", 0.003: "#0d366b"}
INSTANT_COL = "#eb6834"
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.6,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "legend.frameon": False,
})
def save_fig(fig, name):
    fig.savefig(EXP_DIR / f"{name}.jpg", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# ===== Experiment 10 core =====
"""Experiment 10 core: two-compartment plaque with daily home care nested in professional recall.

Plaque signal  u(t) = u_a(t) + u_r(t)
  accessible  u_a: du_a/dt = k_a (U_a - u_a); at every brushing (once per day, period DAY)
                   u_a -> (1 - eta) u_a
  retained    u_r: du_r/dt = k_r (U_r - u_r); untouched by home care
At every professional visit (period T): u_a, u_r -> 0 (supragingival cleaning) and,
optionally, M -> 0 (subgingival instrumentation).

The forcing is exogenous, so it is tabulated for one recall interval at the three RK4
stage points of every step (left limit at the end of a step, so the daily drops and the
recall reset fall exactly on step boundaries).

'averaged' forcing replaces u_a by its daily mean, linearly interpolated between day
centres (anchored at 0 at the recall reset): the smooth envelope assumed in Exp 07.
"""
import numpy as np

COMPARATOR = dict(n=4.0, q=4.0, K=0.9, th0=0.8, r=1.0, a=1.0)
DT = 0.1
DAY = 0.6          # 1 model time unit ~ 1.7 days; T = 60 ~ 100 days (~3-monthly recall)
T_RECALL = 60.0
K_R = 0.05         # retained plaque: regrowth time 1/k_r = 20 units ~ 5 weeks (Exp 07 value)
K_A = 0.5          # accessible plaque: regrowth time 2 units ~ 3 days
ENTRAP_H = 0.5


def hill(x, h, n):
    x = np.maximum(x, 0.0)
    xn = x ** n
    return xn / (xn + h ** n)


def forcing_tables(p_r, U_a, eta, variant='explicit', T=T_RECALL, day=DAY, k_r=K_R, k_a=K_A, dt=DT, fine=20):
    """Return (U0, Uh, U1, ua_fine, ur_fine, t_fine) for one recall interval.
    p_r: retained-plaque peak reached before recall, U_r = p_r / (1 - exp(-k_r T))."""
    Ns = int(round(T / dt)); n_day = int(round(day / dt)); assert abs(n_day * dt - day) < 1e-12
    U_r = p_r / (1 - np.exp(-k_r * T))
    tf = np.linspace(0, T, Ns * fine + 1)                       # fine grid for averaging and plots
    ur = U_r * (1 - np.exp(-k_r * tf))
    # accessible plaque: exact within each day, drop at day starts (after the first)
    ua = np.zeros_like(tf); start_val = 0.0; pre = []
    days = int(round(T / day))
    for m in range(days):
        i0, i1 = m * n_day * fine, (m + 1) * n_day * fine
        tt = tf[i0:i1 + 1] - tf[i0]
        seg = U_a - (U_a - start_val) * np.exp(-k_a * tt)
        ua[i0:i1 + 1] = seg                                       # boundary keeps the next day's post-brushing value
        pre.append(seg[-1])
        start_val = (1 - eta) * seg[-1]                          # brushing at the start of the next day
    if variant == 'averaged':
        centres, means = [0.0], [0.0]
        for m in range(days):
            i0, i1 = m * n_day * fine, (m + 1) * n_day * fine
            centres.append((m + 0.5) * day); means.append(np.trapezoid(ua[i0:i1 + 1], tf[i0:i1 + 1]) / day)
        ua = np.interp(tf, centres, means)
    u = ua + ur
    # stage values per step j: right limit at start, midpoint, left limit at end
    U0 = np.empty(Ns); Uh = np.empty(Ns); U1 = np.empty(Ns)
    for j in range(Ns):
        a, h, b = j * fine, j * fine + fine // 2, (j + 1) * fine
        U0[j] = u[a]; Uh[j] = u[h]; U1[j] = u[b]
        if variant == 'explicit' and (j + 1) % n_day == 0 and (j + 1) < Ns:
            U1[j] = ur[b] + pre[(j + 1) // n_day - 1]           # left limit: plaque just before brushing
    U0[0] = 0.0
    return U0, Uh, U1, ua, ur, tf


def _rhs(H, M, A, u, eps, p, mode):
    A_eff = p['a'] * H if mode == 'instant' else A
    dH = -H + hill(M + u, p['th0'] * np.exp(-A_eff), p['n'])
    dM = p['r'] * (-M + hill(H, p['K'], p['q']))
    dA = eps * (p['a'] * H - A) if mode == 'adaptive' else 0.0 * A
    return dH, dM, dA


def simulate(tables, eps, n_cycles, mode='adaptive', reset_M=False, p=COMPARATOR, dt=DT, y0=None, record_trace=None):
    """tables: (U0, Uh, U1) arrays of shape (n_el, Ns).  Returns per-cycle Hmin, Hmax, theta at recall,
    relapse_time (first rise of H through 0.5 after the recall), and frac_high."""
    U0, Uh, U1 = [np.atleast_2d(x) for x in tables]
    n_el, Ns = U0.shape
    H = np.zeros(n_el); M = np.zeros(n_el); A = np.zeros(n_el)
    if y0 is not None:
        H, M, A = [np.array(np.broadcast_to(v, (n_el,)), float) for v in y0]
    eps = np.broadcast_to(np.asarray(eps, float), (n_el,))
    rec = {k: np.full((n_el, n_cycles), np.nan) for k in ('Hmin', 'Hmax', 'theta_recall', 'relapse_time', 'frac_high')}
    for c in range(n_cycles):
        if reset_M and c > 0:
            M = np.zeros(n_el)
        rec['theta_recall'][:, c] = p['th0'] * np.exp(-(p['a'] * H if mode == 'instant' else A))
        Hmin, Hmax = H.copy(), H.copy(); t_rel = np.full(n_el, np.nan); Thigh = np.zeros(n_el)
        for j in range(Ns):
            f = lambda h, m, a, u: _rhs(h, m, a, u, eps, p, mode)
            k1 = f(H, M, A, U0[:, j])
            k2 = f(H + .5*dt*k1[0], M + .5*dt*k1[1], A + .5*dt*k1[2], Uh[:, j])
            k3 = f(H + .5*dt*k2[0], M + .5*dt*k2[1], A + .5*dt*k2[2], Uh[:, j])
            k4 = f(H + dt*k3[0], M + dt*k3[1], A + dt*k3[2], U1[:, j])
            Hn = H + dt/6 * (k1[0] + 2*k2[0] + 2*k3[0] + k4[0])
            M = M + dt/6 * (k1[1] + 2*k2[1] + 2*k3[1] + k4[1])
            A = A + dt/6 * (k1[2] + 2*k2[2] + 2*k3[2] + k4[2])
            newly = np.isnan(t_rel) & (H <= ENTRAP_H) & (Hn > ENTRAP_H)
            t_rel = np.where(newly, (j + 1) * dt, t_rel)
            Thigh += dt * (Hn > ENTRAP_H)
            H = Hn; Hmin = np.minimum(Hmin, H); Hmax = np.maximum(Hmax, H)
            if record_trace is not None:
                record_trace(c, j, H, M, A)
        rec['Hmin'][:, c], rec['Hmax'][:, c] = Hmin, Hmax
        rec['relapse_time'][:, c], rec['frac_high'][:, c] = t_rel, Thigh / (Ns * dt)
    return rec


def conversion_cycle(Hmin):
    out = np.full(Hmin.shape[0], np.nan)
    for e in range(Hmin.shape[0]):
        row = Hmin[e] > ENTRAP_H
        if row[-1]:
            non = np.where(~row)[0]; out[e] = 0 if non.size == 0 else non[-1] + 1
    return out

In [ ]:
# =====================================================================
# PARAMETERS
# =====================================================================
ETAS = [0.3, 0.5, 0.7, 0.9]           # daily home-care efficacy (fraction of accessible plaque removed)
UAS = [0.3, 0.6]                      # ceiling of accessible plaque
HOSTS = [('instant', 0.0, 30), ('adaptive', 0.03, 40), ('adaptive', 0.01, 70)]   # (mode, eps, cycles)
PK = np.round(np.arange(0.05, 0.7001, 0.005), 4)                                  # retained-plaque peak grid
if QUICK:
    ETAS = [0.3, 0.9]; UAS = [0.3]; HOSTS = HOSTS[:2]; PK = np.round(np.arange(0.05, 0.7001, 0.02), 4)
host_name = lambda mode, eps: 'instant' if mode == 'instant' else f'eps={eps}'
with open(EXP_DIR / "parameters.json", "w") as f:
    json.dump(dict(backbone=COMPARATOR, dt=DT, day=DAY, T_recall=T_RECALL, k_r=K_R, k_a=K_A, etas=ETAS, U_a=UAS,
                   hosts=[host_name(m, e) for m, e, _ in HOSTS], retained_peak_grid=[float(PK[0]), float(PK[-1]), float(PK[1] - PK[0])],
                   time_mapping="1 unit ~ 1.7 days; T = 60 ~ 100 days; 1/k_r ~ 5 weeks; 1/k_a ~ 3 days"), f, indent=4)

import matplotlib.pyplot as plt
def stack_tables(specs):
    """specs: list of (p_r, U_a, eta, variant) -> stacked (U0, Uh, U1)"""
    tt = [forcing_tables(*s)[:3] for s in specs]
    return [np.stack([t[i] for t in tt]) for i in range(3)]

## A. Forcing over one recall interval

In [ ]:
rows = []
for Ua in UAS:
    for eta in ETAS:
        U0, Uh, U1, ua, ur, tf = forcing_tables(0.5, Ua, eta, 'explicit')
        s = (tf > 50) & (tf < 59)
        rows.append(dict(U_a=Ua, eta=eta, daily_min=ua[s].min(), daily_max=ua[s].max(), daily_mean=ua[s].mean(),
                         daily_ripple=ua[s].max() - ua[s].min()))
dfA = pd.DataFrame(rows); dfA.to_csv(EXP_DIR / "accessible_plaque_daily_cycle.csv", index=False); display(dfA.round(3))

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.3), constrained_layout=True)
U0, Uh, U1, ua, ur, tf = forcing_tables(0.5, UAS[0], 0.5, 'explicit')
_, _, _, ua_av, _, _ = forcing_tables(0.5, UAS[0], 0.5, 'averaged')
for a_, xl in [(ax[0], (0, 60)), (ax[1], (20, 26))]:
    a_.plot(tf, ur, color=MEM_COL[0.03], label="retained (missed by home care)")
    a_.plot(tf, ua, color=INK2, lw=0.9, label="accessible, daily brushing")
    a_.plot(tf, ua_av, color=INK, lw=1.2, ls="--", label="accessible, daily mean")
    a_.plot(tf, ua + ur, color=MEM_COL[0.01], lw=1.2, label="total plaque signal")
    a_.set(xlim=xl, xlabel="time within recall interval (1 unit ~ 1.7 days)", ylabel="plaque")
ax[0].set_title(f"One recall interval (U_a = {UAS[0]}, η = 0.5)", loc="left"); ax[1].set_title("Zoom: ten days", loc="left")
ax[0].legend(fontsize=7, loc="center right")
pd.DataFrame(dict(t=tf[::10], accessible=ua[::10], accessible_mean=ua_av[::10], retained=ur[::10])).to_csv(
    EXP_DIR / "fig10_01_forcing_data.csv", index=False)
save_fig(fig, "fig10_01_forcing")

## B. Validation and example trajectories

In [ ]:
# U_a = 0 must reproduce Exp 07 at T = 60 (instant 0.4955, slow memory eps = 0.01 0.5701; 0.005 grid)
val = []
PKv = np.round(np.arange(0.45, 0.62, 0.005), 4)
for mode, eps, ncyc in [('instant', 0.0, 30), ('adaptive', 0.01, 70)]:
    r = simulate(stack_tables([(p, 0.0, 0.5, 'explicit') for p in PKv]), eps, ncyc, mode=mode)
    nc = conversion_cycle(r['Hmin'])
    val.append(dict(host=host_name(mode, eps), threshold_Ua0=PKv[np.argmax(~np.isnan(nc))],
                    exp07_reference={'instant': 0.4955, 'eps=0.01': 0.5701}[host_name(mode, eps)]))
dfV = pd.DataFrame(val); dfV.to_csv(EXP_DIR / "validation_Ua0_vs_exp07.csv", index=False); display(dfV)

# example: same site, explicit vs averaged home care (slow memory), two recall intervals
eps_ex = 0.03 if not QUICK else 0.03
trace = {}
def rec_fn(tag):
    def r(c, j, H, M, A):
        if j % 2 == 0: trace.setdefault(tag, []).append((c * T_RECALL + (j + 1) * DT, H[0], COMPARATOR['th0'] * np.exp(-A[0])))
    return r
for var in ['explicit', 'averaged']:
    simulate(stack_tables([(0.45, UAS[0], 0.5, var)]), eps_ex, 3, record_trace=rec_fn(var))
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.3), constrained_layout=True)
for var, col, ls in [('explicit', MEM_COL[0.01], '-'), ('averaged', INK, '--')]:
    a = np.array(trace[var])
    ax[0].plot(a[:, 0], a[:, 1], ls, color=col, lw=1.0, label=f"{var} home care")
    ax[1].plot(a[:, 0], a[:, 2], ls, color=col, lw=1.0, label=f"{var} home care")
    pd.DataFrame(a, columns=['t', 'H', 'theta']).to_csv(EXP_DIR / f"fig10_02_example_{var}.csv", index=False)
ax[0].set(xlabel="time", ylabel="host response H", title=f"Response (ε = {eps_ex}, p_r = 0.45)")
ax[1].set(xlabel="time", ylabel="threshold θ", title="Memory")
ax[0].legend(fontsize=7)
ax[0].set_ylim(-0.02, 0.56)
ins = ax[0].inset_axes([0.42, 0.70, 0.24, 0.22])
for var, col, ls in [('explicit', MEM_COL[0.01], '-'), ('averaged', INK, '--')]:
    a = np.array(trace[var]); m = (a[:, 0] > 140) & (a[:, 0] < 146); ins.plot(a[m, 0], a[m, 1], ls, color=col, lw=0.9)
ins.set_title("ten days (t = 140–146)", fontsize=7); ins.tick_params(labelsize=6)
save_fig(fig, "fig10_02_example_trajectories")

## C. Thresholds and latency: explicit vs averaged home care

In [ ]:
combos = [(Ua, eta, var) for Ua in UAS for eta in ETAS for var in ['explicit', 'averaged']]
tabs = stack_tables([(p, Ua, eta, var) for (Ua, eta, var) in combos for p in PK])
rows = []
for mode, eps, ncyc in HOSTS:
    t0 = time.time()
    r = simulate(tabs, eps, ncyc, mode=mode)
    nc = conversion_cycle(r['Hmin']).reshape(len(combos), len(PK))
    for k, (Ua, eta, var) in enumerate(combos):
        for ip, p in enumerate(PK):
            rows.append(dict(host=host_name(mode, eps), U_a=Ua, eta=eta, variant=var, p_r=p, cycles_to_conversion=nc[k, ip]))
    print(f"{host_name(mode, eps):9s} done ({time.time() - t0:.0f}s)")
dfC = pd.DataFrame(rows); dfC.to_csv(EXP_DIR / "fig10_03_conversion_grid_data.csv", index=False)
mean_acc = dfA.set_index(['U_a', 'eta']).daily_mean
thr = (dfC[dfC.cycles_to_conversion.notna()].groupby(['host', 'U_a', 'eta', 'variant']).p_r.min()
       .rename('threshold_retained_peak').reset_index())
thr['accessible_daily_mean'] = [mean_acc.loc[(u, e)] for u, e in zip(thr.U_a, thr.eta)]
thr.to_csv(EXP_DIR / "thresholds_explicit_vs_averaged.csv", index=False)
cmp_ = thr.pivot_table(index=['host', 'U_a', 'eta'], columns='variant', values='threshold_retained_peak')
cmp_['difference'] = cmp_['explicit'] - cmp_['averaged']
display(cmp_.round(4))

In [ ]:
# latency comparison: cycles to conversion at fixed distances above each variant's own threshold
lat = []
for (h, Ua, eta), g in dfC.groupby(['host', 'U_a', 'eta']):
    for var in ['explicit', 'averaged']:
        gg = g[g.variant == var].sort_values('p_r'); ok = gg.cycles_to_conversion.notna().values
        if not ok.any(): continue
        p0 = gg.p_r.values[np.argmax(ok)]
        for dp in [0.01, 0.02, 0.05, 0.10]:
            v = gg[np.isclose(gg.p_r, p0 + dp)].cycles_to_conversion
            lat.append(dict(host=h, U_a=Ua, eta=eta, variant=var, above_threshold=dp, cycles=v.iloc[0] if len(v) else np.nan))
dfL = pd.DataFrame(lat); dfL.to_csv(EXP_DIR / "latency_explicit_vs_averaged.csv", index=False)
display(dfL.pivot_table(index=['host', 'U_a', 'eta', 'above_threshold'], columns='variant', values='cycles'))

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.5), constrained_layout=True)
for (mode, eps, _) in HOSTS:
    h = host_name(mode, eps); col = INSTANT_COL if mode == 'instant' else MEM_COL[eps]
    for var, mk, fc in [('explicit', 'o', col), ('averaged', 's', 'none')]:
        g = thr[(thr.host == h) & (thr.variant == var)]
        ax[0].plot(g.accessible_daily_mean, g.threshold_retained_peak, mk, color=col, mfc=fc, ms=4,
                   label=f"{h}, {var}" if var == 'explicit' else None)
    gg = thr[(thr.host == h) & (thr.variant == 'explicit')].sort_values('accessible_daily_mean')
    if len(gg) > 1:
        cf = np.polyfit(gg.accessible_daily_mean, gg.threshold_retained_peak, 1)
        xx = np.linspace(0, gg.accessible_daily_mean.max() * 1.05, 20); ax[0].plot(xx, np.polyval(cf, xx), color=col, lw=0.8)
ax[0].set(xlabel="daily-mean accessible plaque (set by home care)", ylabel="tolerable retained-plaque peak",
          title="Home care acts through its daily mean")
ax[0].plot([], [], 's', color=INK2, mfc='none', label="open: averaged home care"); ax[0].legend(fontsize=6.5)
for (mode, eps, _) in HOSTS:
    if mode == 'instant': continue
    h = host_name(mode, eps)
    for var, ls in [('explicit', '-'), ('averaged', '--')]:
        g = dfC[(dfC.host == h) & (dfC.U_a == UAS[0]) & (dfC.eta == ETAS[len(ETAS) // 2]) & (dfC.variant == var)]
        ax[1].plot(g.p_r, g.cycles_to_conversion, ls, color=MEM_COL[eps], label=f"{h}, {var}")
ax[1].set(yscale='log', xlabel="retained-plaque peak p_r", ylabel="cycles before conversion",
          title=f"Latency (U_a = {UAS[0]}, η = {ETAS[len(ETAS) // 2]})")
ax[1].legend(fontsize=6.5)
save_fig(fig, "fig10_03_thresholds_and_latency")

## D. Debridement at recall with daily home care

At every recall, both plaque pools and $M$ are reset. The outcome is the relapse-free fraction of each interval, explicit vs averaged home care, compared with Exp 07 (Fig 07_05).

In [ ]:
Ua_d, eta_d = UAS[0], 0.5
PRS = [0.50, 0.65]; NCD = 80 if not QUICK else 20
rows = []
fig, ax = plt.subplots(1, len(PRS), figsize=(9.5, 3.3), constrained_layout=True, sharey=True)
for a_i, pr in enumerate(PRS):
    for (mode, eps, _) in HOSTS:
        h = host_name(mode, eps); col = INSTANT_COL if mode == 'instant' else MEM_COL[eps]
        for var, ls in [('explicit', '-'), ('averaged', '--')]:
            r = simulate(stack_tables([(pr, Ua_d, eta_d, var)]), eps, NCD, mode=mode, reset_M=True)
            rt, fh = r['relapse_time'][0], r['frac_high'][0]
            rf = np.where(np.isnan(rt), np.where(fh > 0, 0.0, 1.0), rt / T_RECALL)
            ax[a_i].plot(np.arange(NCD), rf, ls, color=col, lw=1.1 if var == 'explicit' else 1.0,
                         label=f"{h}, {var}")
            rows += [dict(host=h, variant=var, p_r=pr, visit=c, relapse_free_fraction=rf[c]) for c in range(NCD)]
    ax[a_i].set(xlabel="recall visit with debridement", title=f"retained peak {pr} (U_a = {Ua_d}, η = {eta_d})")
ax[0].set_ylabel("relapse-free fraction of interval"); ax[0].legend(fontsize=6.5)
dfD = pd.DataFrame(rows); dfD.to_csv(EXP_DIR / "fig10_04_debridement_data.csv", index=False)
save_fig(fig, "fig10_04_debridement")
dd = dfD.pivot_table(index=['host', 'p_r', 'visit'], columns='variant', values='relapse_free_fraction')
print("max |explicit - averaged| relapse-free fraction:", float(np.nanmax(np.abs(dd.explicit - dd.averaged))))

## E. Checks and summary

In [ ]:
checks = []
checks.append(dict(check="U_a = 0 reproduces Exp 07 thresholds (within 0.005 grid)",
                   value=bool(np.all(np.abs(dfV.threshold_Ua0 - dfV.exp07_reference) <= 0.005 + 1e-9))))
step = float(PK[1] - PK[0])
checks.append(dict(check="max |threshold explicit - averaged|", value=float(np.nanmax(np.abs(cmp_['difference'])))))
checks.append(dict(check="grid step", value=step))
mono = all(np.all(np.diff(g.sort_values('p_r').cycles_to_conversion.notna().astype(int)) >= 0)
           for _, g in dfC.groupby(['host', 'U_a', 'eta', 'variant']))
checks.append(dict(check="conversion monotone in retained peak", value=mono))
for (mode, eps, _) in HOSTS:
    h = host_name(mode, eps); g = thr[(thr.host == h) & (thr.variant == 'explicit')]
    if len(g) > 2:
        cf = np.polyfit(g.accessible_daily_mean, g.threshold_retained_peak, 1)
        res = g.threshold_retained_peak - np.polyval(cf, g.accessible_daily_mean)
        checks.append(dict(check=f"{h}: threshold = a + b x daily-mean accessible plaque; slope b", value=cf[0]))
        checks.append(dict(check=f"{h}: max |residual| of that line (both U_a)", value=float(np.abs(res).max())))
dfE = pd.DataFrame(checks); dfE.to_csv(EXP_DIR / "checks_and_summary.csv", index=False); display(dfE)

In [ ]:
readme = """Experiment 10 - Daily home care nested in professional recall
=============================================================
Plaque u = u_a + u_r. Accessible u_a: regrowth 1/k_a = 2 units (~3 days), removed daily with
efficacy eta. Retained u_r: k_r = 0.05 (Exp 07), removed only at recall (T = 60 ~ 100 days).
Explicit daily drops vs day-averaged envelope (the Exp 07 assumption). Hosts: instant adaptation,
slow memory eps = 0.03, 0.01. Comparator backbone (Exp 06B).

Files
- parameters.json, software_versions.txt
- accessible_plaque_daily_cycle.csv            daily min / max / mean of accessible plaque by (U_a, eta)
- fig10_01_forcing                              one recall interval and a ten-day zoom
- validation_Ua0_vs_exp07.csv                   U_a = 0 reproduces Exp 07 thresholds
- fig10_02_example_trajectories                 H and theta, explicit vs averaged home care
- fig10_03_thresholds_and_latency, fig10_03_conversion_grid_data.csv,
  thresholds_explicit_vs_averaged.csv, latency_explicit_vs_averaged.csv
- fig10_04_debridement                          relapse-free fraction per recall with debridement
- checks_and_summary.csv
"""
with open(EXP_DIR / "README.txt", "w") as f:
    f.write(readme)
archive = shutil.make_archive(str(BASE_DIR / "Experiment_10_Daily_Home_Care_EXPORT"), 'zip', EXP_DIR)
print("Saved:", archive)